# Lab 2 · LLM Workflows: composition you control

An agent lets the model decide the steps. A workflow is where **you** decide the steps and the model fills each one. Most production LLM systems are workflows, not agents, because determinism is a feature when bookings and money are on the line.

This lab builds the five workflow patterns that cover the large majority of real systems. Each one gets the same treatment: the intuition, a diagram, running code on TravelMind, when to reach for it, and a skeptic's view of where it bites.

**The five patterns**

| Pattern | One line | What you control |
|---|---|---|
| Prompt chaining | steps in a fixed sequence | the order |
| Routing | classify, then branch | the paths |
| Parallelization | run independent calls at once | the fan-out |
| Orchestrator-workers | plan subtasks, run them, merge | the decomposition |
| Evaluator-optimizer | generate, grade, refine, repeat | the quality bar |

## The map of workflows

```mermaid
graph TD
    START["A task arrives"] --> Q{"How structured is it?"}
    Q -->|"one fixed line of steps"| CHAIN["Prompt chaining"]
    Q -->|"path depends on input"| ROUTE["Routing"]
    Q -->|"many independent parts"| PAR["Parallelization"]
    Q -->|"parts unknown up front"| ORCH["Orchestrator-workers"]
    Q -->|"needs a quality loop"| EVAL["Evaluator-optimizer"]
```

Read the diamond as the real question you ask before writing any workflow: how much structure does this task actually have?

## First, the fork: workflow or agent?

Before choosing a pattern, choose a family. This is the single most consequential decision in applied LLM work, and getting it wrong is the most common cause of slow, flaky, expensive systems.

| Dimension | Workflow | Agent |
|---|---|---|
| Who picks the steps | you | the model |
| Path | fixed, or a branch you defined | decided at runtime |
| Predictability | high | lower |
| Cost and latency | bounded | open-ended |
| Debugging | straightforward | harder |
| Best for | known procedures | open-ended goals |

```mermaid
graph TD
    G["You have a goal"] --> K{"Do you know the steps in advance?"}
    K -->|"yes"| W["Build a workflow, this lab"]
    K -->|"no, the model must decide"| A["Build an agent, Lab 3"]
    W --> W2{"Same steps every time?"}
    W2 -->|"yes"| C["Chaining or parallelization"]
    W2 -->|"path depends on input"| R["Routing or orchestrator-workers"]
```

> **Skeptic's rule.** Reach for an agent last, not first. If a workflow can express the task, it will be cheaper, faster, and easier to trust. Agents earn their overhead only when the steps genuinely cannot be known ahead of time.

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
llm = ChatBedrockConverse(model_id=MODEL_ID, region_name="us-east-1", temperature=0.2)

## Pattern 1 · Prompt chaining

**Feel it.** One big prompt that does five things at once does all five poorly. Split the work into a line of small, sharp steps, and each step gets easy.

Chaining runs steps in a fixed order, passing each output into the next. Use it when a task decomposes into clear sequential stages.

```mermaid
graph LR
    IN["passenger complaint"] --> S1["extract the core issue"]
    S1 --> S2["classify urgency"]
    S2 --> S3["draft a matched reply"]
    S3 --> OUT["reply"]
```

In [ ]:
extract = ChatPromptTemplate.from_messages([
    ("system", "Extract the passenger's core issue in one sentence."),
    ("human", "{complaint}"),
]) | llm | StrOutputParser()

classify = ChatPromptTemplate.from_messages([
    ("system", "Label urgency as low, medium, or high. Reply with only the label."),
    ("human", "Issue: {issue}"),
]) | llm | StrOutputParser()

draft = ChatPromptTemplate.from_messages([
    ("system", "Draft a reply as TravelMind. Match the tone to the urgency."),
    ("human", "Issue: {issue}\nUrgency: {urgency}"),
]) | llm | StrOutputParser()

# ({key: runnable}) -> automatically converts into a RunnableParallel
chain = (
    {"issue": extract} 
    | RunnablePassthrough.assign(urgency=classify)
    | draft
)

print(chain.invoke({
    "complaint": "My BLR to DEL flight JX48Q2 was cancelled and I have a wedding to reach tonight."
}))

# TravelMind Response

**Subject: Immediate Action Required – Your Bangalore to Delhi Flight Cancelled**

---

I understand this is urgent—missing a wedding tonight is not an option. Let me help you find solutions **right now**.

## Immediate Actions (Next 30 minutes):

1. **Alternative Flights**
   - Check other airlines departing Bangalore today (IndiGo, SpiceJet, Vistara, Air India)
   - Look at nearby airports: Hyderabad (1.5 hrs away) or Chennai (2 hrs away)
   - Set up alerts on Google Flights & Skyscanner for real-time availability

2. **Ground Transport Options**
   - **Drive**: Bangalore to Delhi is ~2,000 km (24-26 hours non-stop) — not viable for tonight
   - **Train**: Check IRCTC for emergency bookings on tonight's trains (though unlikely to have seats)
   - **Bus**: Premium overnight buses (Redbus, Ixigo) — arrives tomorrow morning (not suitable)

3. **Backup Plan**
   - Contact the wedding organizers immediately — explain the situation
   - Ask if you can join virtually 

### Walkthrough

| Stage | Input | Output |
|---|---|---|
| `{"issue": extract}` | `{"complaint": ...}` | `{"issue": ...}` |
| `.assign(urgency=classify)` | `{"issue": ...}` | `{"issue": ..., "urgency": ...}` |
| `draft` | `{"issue": ..., "urgency": ...}` | the reply |

`RunnablePassthrough.assign` keeps what it already has and adds a new key. That is how state accumulates down a chain.

**Reach for it when** the task is a clean sequence and each step benefits from a focused prompt.

> **Skeptic's view.** Chaining adds one round trip per step, so latency stacks. An early misread cascades: if `extract` gets the issue wrong, everything after inherits the mistake. Keep chains short, and put the highest-signal step first.

## Pattern 2 · Routing

**Feel it.** A refund question and a baggage question need different handling. Forcing one prompt to cover both makes it vague. Classify first, then send each request down a specialized path.

Routing is two moves: a classifier that reads the input, and a set of specialized chains it dispatches to.

```mermaid
graph TD
    IN["request"] --> CL["classifier"]
    CL --> D{"category"}
    D -->|"rebooking"| RB["rebooking chain"]
    D -->|"refund"| RF["refund chain"]
    D -->|"baggage"| BG["baggage chain"]
    D -->|"other"| DF["default chain"]
```

In [3]:
class Intent(BaseModel):
    category: str = Field(description="one of: rebooking, refund, baggage, other")

classifier = ChatPromptTemplate.from_messages([
    ("system", "Classify the request into exactly one of: rebooking, refund, baggage, other."),
    ("human", "{request}"),
]) | llm.with_structured_output(Intent)

def specialist(role):
    return ChatPromptTemplate.from_messages([
        ("system", f"You are TravelMind handling {role}. Answer in 2 sentences."),
        ("human", "{request}"),
    ]) | llm | StrOutputParser()

routes = {
    "rebooking": specialist("rebookings"),
    "refund": specialist("refunds"),
    "baggage": specialist("baggage"),
}
default_chain = specialist("general support")

def pick_and_run(inp):
    category = classifier.invoke({"request": inp["request"]}).category
    chosen = routes.get(category, default_chain)
    return {"category": category, "reply": chosen.invoke(inp)}

router = RunnableLambda(pick_and_run)

for req in [
    "I need my bag traced, it did not arrive at DEL.",
    "Please move JX48Q2 to the morning flight.",
]:
    print(router.invoke({"request": req}))
    print("---")

{'category': 'baggage', 'reply': "I understand your bag didn't arrive in Delhi—I'm sorry for this inconvenience. Please provide me with your booking reference or baggage tag number so I can trace your bag and help locate it for you."}
---
{'category': 'rebooking', 'reply': "I'd be happy to help move booking JX48Q2 to a morning flight. Could you please provide the date of travel and your departure/destination cities so I can check available morning flight options and process the rebooking?"}
---


### Walkthrough

| Piece | Role |
|---|---|
| `Intent` + `with_structured_output` | forces the classifier to return one clean label |
| `routes` dict | maps each label to a specialized chain |
| `pick_and_run` | classify, look up the chain, run it |
| `RunnableLambda` | wraps the Python function so it composes like any runnable |

**Reach for it when** different input types need genuinely different handling, and the set of types is known.

> **Skeptic's view.** The whole workflow is hostage to the classifier. A wrong label sends the request down the wrong path silently. Keep the label set small, add an `other` route as a safety net, and log the category so you can measure misroutes.

## Pattern 3 · Parallelization

**Feel it.** Three independent lookups do not need to wait in line. Fire them together and merge. Wall-clock time drops to the slowest one, not the sum.

Two flavors. **Sectioning** splits a task into independent parts, each handled once. **Voting** runs the same task several times and aggregates for reliability.

```mermaid
graph TD
    IN["cancelled-flight brief"] --> F["fan out"]
    F --> A["policy check"]
    F --> B["fare handling"]
    F --> C["rebooking options"]
    A --> M["merge"]
    B --> M
    C --> M
    M --> OUT["assembled answer"]
```

In [4]:
policy = ChatPromptTemplate.from_messages([
    ("system", "State the cancellation policy for Gold tier in 2 lines."),
    ("human", "{brief}"),
]) | llm | StrOutputParser()

fare = ChatPromptTemplate.from_messages([
    ("system", "Explain fare-difference handling for an involuntary change in 2 lines."),
    ("human", "{brief}"),
]) | llm | StrOutputParser()

options = ChatPromptTemplate.from_messages([
    ("system", "List two concrete rebooking options."),
    ("human", "{brief}"),
]) | llm | StrOutputParser()

sections = RunnableParallel(policy=policy, fare=fare, options=options)

result = sections.invoke({"brief": "PNR JX48Q2, BLR to DEL cancelled, Gold tier."})
for k, v in result.items():
    print(k.upper()); print(v); print("---")

POLICY
# Cancellation Policy - Gold Tier

**Gold tier members receive a full refund for flight cancellations.** Refunds are processed within 5-7 business days to the original payment method.
---
FARE
# Fare Difference Handling - Involuntary Change

**PNR: JX48Q2 | Route: BLR-DEL | Status: Gold Tier**

## Fare Difference Policy

**Gold tier members receive:**

1. **Rebooking on next available flight at no additional cost** - If the new flight is in the same cabin class, no fare difference applies

2. **If upgraded to higher cabin** - Gold tier absorbs the difference; no charge to passenger

3. **If downgraded to lower cabin** - Full refund of fare difference to original payment method within 7-10 business days

---

**Action Required:** Confirm preferred rebooking option and provide contact details for any refund processing.
---
OPTIONS
# Rebooking Options for PNR JX48Q2

**Flight Details:**
- Route: Bangalore (BLR) → Delhi (DEL)
- Status: Cancelled
- Passenger Tier: Gold

## Two Concre

### Sectioning vs voting

| Flavor | What runs | Use for |
|---|---|---|
| Sectioning | different subtasks, once each | independent parts of one answer |
| Voting | the same task several times | reliability on a judgment call |

`RunnableParallel` runs its branches concurrently and returns a dict keyed by branch name. A dict literal inside a pipe does the same thing.

**Reach for it when** subtasks do not depend on each other, or when repeated sampling buys confidence.

> **Skeptic's view.** Parallel does not mean free. You pay for every branch in tokens, and concurrency stresses rate limits. Split only parts that are truly independent; forcing a dependent step into a parallel branch just hides a sequencing bug.

## Pattern 4 · Orchestrator-workers

**Feel it.** Sometimes you do not know the subtasks until you see the input. A complex complaint might need two steps or five. An orchestrator reads the task, decides the subtasks, workers run them, and a synthesizer merges.

The difference from routing: routing picks one path from a fixed menu. Orchestrator-workers generates the subtasks on the fly.

```mermaid
graph TD
    IN["complex request"] --> O["orchestrator: plan subtasks"]
    O --> W1["worker"]
    O --> W2["worker"]
    O --> W3["worker"]
    W1 --> S["synthesizer: merge"]
    W2 --> S
    W3 --> S
    S --> OUT["single answer"]
```

In [5]:
class Plan(BaseModel):
    subtasks: list[str] = Field(description="3 concrete subtasks to resolve the request")

orchestrator = ChatPromptTemplate.from_messages([
    ("system", "Break the request into exactly 3 concrete subtasks for a support specialist."),
    ("human", "{request}"),
]) | llm.with_structured_output(Plan)

worker = ChatPromptTemplate.from_messages([
    ("system", "You are a TravelMind specialist. Complete this subtask in 2 sentences."),
    ("human", "{task}"),
]) | llm | StrOutputParser()

synthesizer = ChatPromptTemplate.from_messages([
    ("system", "Combine the subtask results into one clear reply to the passenger."),
    ("human", "{joined}"),
]) | llm | StrOutputParser()

request = "JX48Q2 cancelled, Gold tier, missed onward connection, wants compensation and a fast rebooking."

plan = orchestrator.invoke({"request": request})
results = worker.batch([{"task": t} for t in plan.subtasks])   # workers run in parallel
joined = "\n\n".join(f"Subtask: {t}\nResult: {r}" for t, r in zip(plan.subtasks, results))
final = synthesizer.invoke({"joined": joined})

print("PLAN:", plan.subtasks)
print("---")
print(final)

PLAN: ["Verify the cancellation of flight JX48Q2 and confirm the passenger's Gold tier status and eligibility for compensation under applicable regulations", 'Process compensation claim for the missed onward connection and arrange expedited rebooking on the next available flight to the final destination', 'Provide the passenger with confirmation of compensation amount, new booking details, and any additional support (meal vouchers, accommodation if needed, priority customer service contact)']
---
# Complete Resolution Summary

Thank you for your patience. Here's your complete resolution for flight JX48Q2 and your missed connection:

## ✓ Flight Cancellation & Compensation Eligibility Confirmed
Your Gold tier status qualifies you for priority compensation processing under EU261/2004 regulations. You are entitled to **€250 in compensation** for this flight disruption, which will be processed to your original payment method within 14 business days.

## ✓ Expedited Rebooking Arranged
We've

### Walkthrough

| Stage | Move |
|---|---|
| orchestrator | returns a typed `Plan` of subtasks, decided from the input |
| `worker.batch(...)` | runs all subtasks concurrently, one call each |
| synthesizer | folds the results into one reply |

**Reach for it when** the number and shape of subtasks vary by input, but the overall procedure (plan, run, merge) stays fixed.

> **Skeptic's view.** This looks like an agent and is often mistaken for one. The difference: control flow here is fixed, only the subtasks vary. The moment you want the model to also decide whether to loop, call tools, and rewrite the plan, you have crossed into agent territory. Go to Lab 3 rather than bolting more logic onto this.

## Pattern 5 · Evaluator-optimizer

**Feel it.** First drafts are rarely the best drafts. Add a critic. One model generates, another grades against a bar, and the generator revises until it passes or hits a cap.

```mermaid
graph TD
    B["brief"] --> G["generator: draft"]
    G --> E{"evaluator: passes the bar?"}
    E -->|"no, with feedback"| G
    E -->|"yes"| OUT["final draft"]
```

In [6]:
class Grade(BaseModel):
    passed: bool = Field(description="true only if tone and completeness are both strong")
    feedback: str = Field(description="one concrete improvement if not passed")

generator = ChatPromptTemplate.from_messages([
    ("system", "Draft a concise, empathetic TravelMind reply. Apply the feedback if any is given."),
    ("human", "Brief: {brief}\nFeedback so far: {feedback}"),
]) | llm | StrOutputParser()

evaluator = ChatPromptTemplate.from_messages([
    ("system", "Grade the reply for tone and completeness."),
    ("human", "{draft}"),
]) | llm.with_structured_output(Grade)

brief = "PNR JX48Q2, Gold tier, BLR to DEL cancelled. Offer rebooking and acknowledge the disruption."
feedback = "none"

for attempt in range(1, 4):
    draft = generator.invoke({"brief": brief, "feedback": feedback})
    grade = evaluator.invoke({"draft": draft})
    print(f"attempt {attempt} | passed={grade.passed}")
    if grade.passed:
        break
    feedback = grade.feedback

print("---FINAL---")
print(draft)

attempt 1 | passed=True
---FINAL---
**Subject: Your Flight BLR-DEL (PNR: JX48Q2) – Rebooking Options Available**

Dear Valued Gold Member,

We sincerely apologize for the cancellation of your flight from Bangalore to Delhi. We understand how disruptive this is to your plans, and we truly appreciate your patience.

**Here's what we're doing for you:**

As a Gold tier member, we're prioritizing your rebooking with the following options:

✈️ **Next available flight** on your preferred date  
✈️ **Alternative dates** within 7 days with no change fees  
✈️ **Full refund** if neither option works for you

**Next steps:**
Please reply with your preferred option, and we'll confirm your new booking within 2 hours. You'll also receive a complimentary meal voucher and lounge access as our apology.

We're here to make this right.

Warm regards,  
**TravelMind Customer Care**  
Available 24/7 | Reference: JX48Q2


### Walkthrough

| Piece | Role |
|---|---|
| generator | drafts, and folds in feedback on later passes |
| evaluator | returns a typed `Grade` with a pass flag and one fix |
| the loop | revises until it passes or hits the cap of 3 |

**Reach for it when** quality is measurable and worth extra calls: tone, correctness against a rubric, format compliance.

> **Skeptic's view.** A loop with no cap is a bill with no ceiling, and returns diminish fast. Cap the attempts. And the evaluator is only as sharp as its rubric; a vague grader rubber-stamps weak drafts. Make the bar concrete.

## Choosing a pattern: the one-look matrix

| If the task is | Pattern | Cost shape | Watch out for |
|---|---|---|---|
| A fixed sequence of stages | Chaining | one call per stage | cascading early errors |
| Different handling by type | Routing | classify plus one branch | misclassification |
| Independent parts, or repeated sampling | Parallelization | one call per branch | token spend, rate limits |
| Subtasks unknown until runtime | Orchestrator-workers | plan, parallel workers, merge | creeping into agent territory |
| Quality worth a critic loop | Evaluator-optimizer | several calls until passed | uncapped loops, weak rubric |

```mermaid
graph TD
    T["task"] --> Q1{"fixed steps?"}
    Q1 -->|"yes, sequential"| CH["Chaining"]
    Q1 -->|"yes, independent"| PA["Parallelization"]
    Q1 -->|"branch by type"| RO["Routing"]
    Q1 -->|"no, subtasks vary"| OR["Orchestrator-workers"]
    T --> Q2{"need a quality bar?"}
    Q2 -->|"yes"| EV["Evaluator-optimizer wraps any of the above"]
```

## When to graduate from LCEL to LangGraph

Everything in this lab runs on LCEL, the pipe. LCEL is ideal for straight-through and lightly branched flows. Reach for LangGraph (Lab 4) when the workflow needs any of these:

| Need | LCEL | LangGraph |
|---|---|---|
| Straight or lightly branched flow | strong | overkill |
| Cycles and revisiting nodes | awkward | native |
| Shared state across many steps | manual | built in |
| Pause, resume, human approval mid-run | no | yes |
| Durability across a crash | no | yes |

The evaluator-optimizer loop above is the tell: the moment you hand-roll a `for` loop with accumulating state, LangGraph is the cleaner home for it.

## Recap

You own the five workflow patterns and, more valuable, the judgment for which is which.

```mermaid
graph LR
    CH["Chaining"] --- RO["Routing"] --- PA["Parallelization"] --- OR["Orchestrator-workers"] --- EV["Evaluator-optimizer"]
```

The through line: **you** hold the control flow. In Lab 3 you hand that control to the model and build an agent, then learn exactly when that trade is worth paying for.

## Exercises

1. **Chain, then measure.** Add a fourth step to the chaining example that translates the final reply into Hindi. Time the chain before and after. Feel the latency cost of each added step.

2. **Route with a fallback drill.** Feed the router a request that fits none of its categories. Confirm it lands on `other`, then add logging of the chosen category.

3. **Sectioning to voting.** Rewrite the parallelization example as voting: run one urgency-classification prompt five times on the same complaint, then take the majority label.

4. **Cap the loop.** In the evaluator-optimizer, set the cap to 1, then to 5. Watch pass rate and call count move together. Decide where the sensible cap sits.

5. **Skeptic's call.** Take the orchestrator-workers example and argue, in two lines, whether it should be an agent instead. State the one condition that would flip your answer.